# VALE3 deep-dive — o único ticker onde o Transformer mostra vida

## Por que este notebook existe

No `expanding_window_cv.ipynb` (5 folds × 5 seeds × 3 tickers), o Transformer + FinBERT perdeu para o dumb baseline em ITUB4 e PETR4 por margens consistentes (~0.25 de AUC). Mas em **VALE3**, o Transformer ocasionalmente bateu o baseline:

| Ticker | Baseline AUC mean | Transformer AUC mean | Δ |
|---|:---:|:---:|:---:|
| ITUB4 | 0.700 | 0.445 | -0.255 |
| PETR4 | 0.702 | 0.447 | -0.255 |
| **VALE3** | **0.599** | **0.635** | **+0.036** |

Pode ser:

- **(a) Sinal real** — sentimento financeiro adiciona valor para mineração mais que para bancos/petróleo. *Se for verdade*, é um achado nuanceado e interessante.
- **(b) Ruído estatístico** — apenas 5 folds × 5 seeds = 25 pontos para VALE3, com std altíssimo. Pode ser variação amostral.

Este notebook decide entre (a) e (b) com:

- **10 folds** (step menor para caber mais janelas) × **10 seeds** = 100 pontos
- Mesmo modelo, mesmas features, mesmo dataset
- Bootstrap CI + teste de Wilcoxon entre os pares (Transformer, baseline) por fold

## Critério de decisão

| Padrão observado | Conclusão |
|---|---|
| Transformer mediana > baseline mediana com Wilcoxon p < 0.05 | Achado real, parcial: "sentimento ajuda mineração". |
| Médias próximas, p > 0.1 | VALE3 foi sorte amostral em 5 folds — unificar tese: baseline vence em todos. |
| Transformer mediana < baseline | Confirma a unificação ainda mais fortemente. |

In [ ]:
import sys, os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import wilcoxon, pearsonr

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from eval_utils import make_binary_target, bootstrap_auc_ci, format_metric_with_ci

TICKER_LABEL = 'VALE3'
YF_TICKER = 'VALE3.SA'
SENTIMENT_CSV = '../4.finbert-br/vale3_daily_sentiment.csv'

HORIZON = 21
WINDOW = 30
MIN_TRAIN_DAYS = 500
VAL_DAYS = 60
TEST_DAYS = 60
STEP_DAYS = 60
N_SEEDS = 10
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

PRICE_COLS = ['Close','Volume','return','ma7','ma21','std21','lag_1','lag_2','lag_3','lag_4','lag_5']
SENT_COLS  = ['n_articles','mean_logit_pos','mean_logit_neg','mean_logit_neu','mean_sentiment']
BASELINE_FEATURES = ['return','lag_1','lag_5','Volume','std21']
FULL_FEATURES = PRICE_COLS + SENT_COLS

## 1. Build dataset (período `max` para ter mais folds)

In [ ]:
def load_prices(yf_ticker, period='max'):
    df = yf.Ticker(yf_ticker).history(period=period, auto_adjust=True).reset_index()
    df['date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
    df = df[['date','Close','Volume']].copy()
    df['return'] = df['Close'].pct_change()
    df['ma7']    = df['Close'].rolling(7).mean()
    df['ma21']   = df['Close'].rolling(21).mean()
    df['std21']  = df['Close'].rolling(21).std()
    for k in range(1, 6):
        df[f'lag_{k}'] = df['Close'].shift(k)
    return df.dropna().reset_index(drop=True)

px = load_prices(YF_TICKER, period='max')
sent = pd.read_csv(SENTIMENT_CSV, parse_dates=['date'])[['date'] + SENT_COLS]
df = px.merge(sent, on='date', how='left').sort_values('date').reset_index(drop=True)
# Restringe ao período onde há sentimento (notícias começam em 2011 para VALE3)
df = df[df['date'] >= sent['date'].min()].reset_index(drop=True)
df[SENT_COLS] = df[SENT_COLS].ffill().fillna(0)
df['target'] = make_binary_target(df['Close'], horizon=HORIZON)
df = df.dropna(subset=['target']).reset_index(drop=True)
print(f'VALE3 dataset: {len(df)} dias | {df["date"].min().date()} → {df["date"].max().date()}')
print(f'  balance up = {df["target"].mean():.3f}')

In [ ]:
def expanding_folds(n_total, min_train=MIN_TRAIN_DAYS, val=VAL_DAYS, test=TEST_DAYS, step=STEP_DAYS):
    folds = []
    train_end = min_train
    while train_end + val + test <= n_total:
        folds.append((train_end, train_end + val, train_end + val + test))
        train_end += step
    return folds

folds = expanding_folds(len(df))
print(f'{len(folds)} folds disponíveis')
for i, fold in enumerate(folds[:5]):
    te, ve, tee = fold
    bal = df.iloc[ve:tee]['target'].mean()
    print(f'  fold {i}: train[0..{te}] val[{te}..{ve}] test[{ve}..{tee}] | test_balance={bal:.3f}')
if len(folds) > 5:
    print(f'  ... e mais {len(folds)-5} folds')

## 2. Modelos

In [ ]:
class Stage4Transformer(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, nlayers=2, window=WINDOW):
        super().__init__()
        self.proj = nn.Linear(n_features, d_model)
        pe = torch.zeros(window, d_model)
        pos = torch.arange(0, window).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=128, dropout=0.2, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 1))
    def forward(self, x):
        h = self.proj(x) + self.pe[:, :x.size(1), :]
        h = self.enc(h)
        return self.head(h.mean(dim=1)).squeeze(-1)

def make_windows(X, y, window=WINDOW):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i]); ys.append(y[i])
    return np.array(Xs), np.array(ys)

def train_transformer_fold(df, fold, seed):
    train_end, val_end, test_end = fold
    train = df.iloc[:train_end]; val = df.iloc[train_end:val_end]; test = df.iloc[val_end:test_end]
    torch.manual_seed(seed); np.random.seed(seed)

    sc = StandardScaler().fit(train[FULL_FEATURES])
    Xtr = sc.transform(train[FULL_FEATURES]); ytr = train['target'].values.astype(int)
    Xva = sc.transform(val[FULL_FEATURES]);   yva = val['target'].values.astype(int)
    Xte = sc.transform(test[FULL_FEATURES]);  yte = test['target'].values.astype(int)
    Xtw, ytw = make_windows(Xtr, ytr)
    Xvw, yvw = make_windows(Xva, yva)
    Xew, yew = make_windows(Xte, yte)
    if len(yew) < 5 or len(np.unique(yew)) < 2:
        return None

    model = Stage4Transformer(n_features=len(FULL_FEATURES)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    pos = (ytw==1).sum(); neg = (ytw==0).sum()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/max(pos,1)], device=device, dtype=torch.float32))

    Xt_t = torch.tensor(Xtw, dtype=torch.float32, device=device)
    yt_t = torch.tensor(ytw, dtype=torch.float32, device=device)
    Xv_t = torch.tensor(Xvw, dtype=torch.float32, device=device)
    yv_t = torch.tensor(yvw, dtype=torch.float32, device=device)
    Xe_t = torch.tensor(Xew, dtype=torch.float32, device=device)

    best=float('inf'); best_state=None; bad=0; patience=15
    for epoch in range(150):
        model.train(); opt.zero_grad()
        loss = loss_fn(model(Xt_t), yt_t); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(Xv_t), yv_t).item()
        if vl < best - 1e-4:
            best=vl; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        y_score = torch.sigmoid(model(Xe_t)).cpu().numpy()
    return {
        'auc': roc_auc_score(yew, y_score),
        'train_balance': float(train['target'].mean()),
        'test_balance':  float(test['target'].mean()),
        'n_test':        int(len(yew)),
    }

def train_baseline_fold(df, fold, seed):
    train_end, val_end, test_end = fold
    train = df.iloc[:train_end]; val = df.iloc[train_end:val_end]; test = df.iloc[val_end:test_end]
    np.random.seed(seed)
    sc = StandardScaler().fit(train[BASELINE_FEATURES])
    Xtr = sc.transform(train[BASELINE_FEATURES]); ytr = train['target'].values.astype(int)
    Xva = sc.transform(val[BASELINE_FEATURES]);   yva = val['target'].values.astype(int)
    Xte = sc.transform(test[BASELINE_FEATURES]);  yte = test['target'].values.astype(int)
    if len(yte) < 5 or len(np.unique(yte)) < 2:
        return None
    pos = (ytr==1).sum(); neg = (ytr==0).sum()
    m = xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=neg/max(pos,1), eval_metric='auc', random_state=seed,
    )
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    y_score = m.predict_proba(Xte)[:,1]
    return {
        'auc': roc_auc_score(yte, y_score),
        'train_balance': float(train['target'].mean()),
        'test_balance':  float(test['target'].mean()),
        'n_test':        int(len(yte)),
    }

## 3. Loop principal — folds × seeds × 2 modelos

In [ ]:
rows = []
t0 = time.time()
for fi, fold in enumerate(folds):
    for seed in range(N_SEEDS):
        rt = train_transformer_fold(df, fold, seed)
        if rt is not None:
            rt.update({'model':'transformer_finbert','fold':fi,'seed':seed}); rows.append(rt)
        rb = train_baseline_fold(df, fold, seed)
        if rb is not None:
            rb.update({'model':'baseline_xgb','fold':fi,'seed':seed}); rows.append(rb)
    if rt is not None:
        shift = rt['test_balance'] - rt['train_balance']
        print(f'  fold {fi:2d} | shift={shift:+.3f} | trans last AUC={rt["auc"]:.3f} | base last AUC={rb["auc"]:.3f} | {time.time()-t0:.0f}s')

results = pd.DataFrame(rows)
results['shift'] = results['test_balance'] - results['train_balance']
results.to_csv('results_vale3_deepdive.csv', index=False)
print(f'\nTotal: {len(results)} runs in {time.time()-t0:.0f}s')

## 4. Agregação por fold + estatística pareada

In [ ]:
fold_agg = results.groupby(['model','fold']).agg(
    auc_mean=('auc','mean'), auc_std=('auc','std'),
    auc_median=('auc','median'),
    train_balance=('train_balance','first'),
    test_balance=('test_balance','first'),
    shift=('shift','first'),
).reset_index()
print(fold_agg.round(3).to_string())

# Pareamento por fold
tr = fold_agg[fold_agg.model=='transformer_finbert'].set_index('fold')['auc_mean']
ba = fold_agg[fold_agg.model=='baseline_xgb'].set_index('fold')['auc_mean']
common = tr.index.intersection(ba.index)
diff = tr.loc[common] - ba.loc[common]
print(f'\nPareamento sobre {len(common)} folds:')
print(f'  Transformer mean over folds: {tr.loc[common].mean():.3f}')
print(f'  Baseline    mean over folds: {ba.loc[common].mean():.3f}')
print(f'  Δ (trans − base) mean: {diff.mean():+.3f} | std: {diff.std():.3f}')
print(f'  Folds onde trans > base: {(diff > 0).sum()}/{len(diff)}')

In [ ]:
# Wilcoxon signed-rank test (não-paramétrico, assume mediana)
if len(diff) >= 5:
    stat, p = wilcoxon(diff)
    print(f'Wilcoxon signed-rank test (Transformer vs baseline, paired by fold):')
    print(f'  W = {stat:.2f}, p = {p:.4f}')
    if p < 0.05:
        side = 'Transformer melhor' if diff.mean() > 0 else 'baseline melhor'
        print(f'  → diferença ESTATISTICAMENTE SIGNIFICATIVA ({side})')
    else:
        print(f'  → diferença NÃO significativa — modelos são equivalentes em VALE3')

# Bootstrap CI sobre a média do delta
rng = np.random.default_rng(42)
boot_means = [diff.iloc[rng.integers(0, len(diff), len(diff))].mean() for _ in range(2000)]
lo, hi = np.quantile(boot_means, [0.025, 0.975])
print(f'\n95% bootstrap CI sobre Δ médio: [{lo:+.3f}, {hi:+.3f}]')
print(f'  CI inclui zero? {"SIM (sem efeito)" if lo <= 0 <= hi else "NÃO (efeito real)"}')

## 5. Visualizações

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Painel 1: AUC ao longo dos folds
ax = axes[0]
ax.errorbar(fold_agg[fold_agg.model=='transformer_finbert']['fold'],
            fold_agg[fold_agg.model=='transformer_finbert']['auc_mean'],
            yerr=fold_agg[fold_agg.model=='transformer_finbert']['auc_std'].fillna(0),
            fmt='o-', color='C0', label='Transformer + FinBERT', capsize=3)
ax.errorbar(fold_agg[fold_agg.model=='baseline_xgb']['fold'],
            fold_agg[fold_agg.model=='baseline_xgb']['auc_mean'],
            yerr=fold_agg[fold_agg.model=='baseline_xgb']['auc_std'].fillna(0),
            fmt='^-', color='C3', label='Dumb baseline (XGB)', capsize=3)
ax.axhline(0.5, ls=':', color='gray', label='Acaso')
ax.set_xlabel('Fold (tempo →)'); ax.set_ylabel('AUC (mean ± std over seeds)')
ax.set_title(f'VALE3 — {len(folds)} folds × {N_SEEDS} seeds')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Painel 2: Δ (trans − base) por fold
ax = axes[1]
colors = ['C2' if d > 0 else 'C7' for d in diff]
ax.bar(diff.index, diff.values, color=colors, edgecolor='k', linewidth=0.5)
ax.axhline(0, color='k', lw=0.8)
ax.axhline(diff.mean(), ls='--', color='C0', label=f'Δ médio = {diff.mean():+.3f}')
ax.set_xlabel('Fold'); ax.set_ylabel('AUC(Transformer) − AUC(baseline)')
ax.set_title('Vantagem do Transformer por fold (verde = trans melhor)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('vale3_deepdive.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# Distribuição de AUC (todos os pontos individuais — folds × seeds)
fig, ax = plt.subplots(figsize=(8, 4.5))
tr_all = results[results.model=='transformer_finbert']['auc']
ba_all = results[results.model=='baseline_xgb']['auc']
ax.hist(tr_all, bins=20, alpha=0.6, label=f'Transformer + FinBERT (mean={tr_all.mean():.3f})', color='C0')
ax.hist(ba_all, bins=20, alpha=0.6, label=f'Dumb baseline (mean={ba_all.mean():.3f})', color='C3')
ax.axvline(0.5, ls=':', color='gray', label='Acaso')
ax.axvline(tr_all.median(), ls='--', color='C0', alpha=0.7)
ax.axvline(ba_all.median(), ls='--', color='C3', alpha=0.7)
ax.set_xlabel('ROC-AUC'); ax.set_ylabel('# runs')
ax.set_title(f'VALE3 — distribuição de {len(tr_all) + len(ba_all)} runs (folds × seeds)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('vale3_deepdive_hist.png', dpi=140, bbox_inches='tight')
plt.show()

## 6. Decisão

Olhe os 3 indicadores principais:

1. **Δ médio entre Transformer e baseline ao longo dos folds** — quão diferente é o desempenho?
2. **Wilcoxon p-value** — a diferença é estatisticamente significativa?
3. **Bootstrap 95% CI sobre Δ** — o CI contém zero?

Cenários:

- **Δ > +0.05, p < 0.05, CI não contém zero** → VALE3 é exceção real. Discutir por que mineração é diferente.
- **|Δ| < 0.03, p > 0.1, CI contém zero** → VALE3 era ruído amostral. Unificar tese: baseline vence em todos.
- **Resultado intermediário** → reportar honestamente como inconclusivo, baseline ainda preferido por estabilidade.